In [21]:
import pandas as pd
import itertools
import requests
from numpy.ma.extras import unique


In [22]:
url = 'https://drive.google.com/uc?id=1IOPTVq2ooQfZRkF3rAjGkTjRtbotG7FF'
response = requests.get(url)
lines = [line for line in response.text.split('\n') if line.strip()]


In [23]:
product_counts = {}
all_pairs = []
for line in lines:
    products = list(set([p.strip() for p in line.split('@@@') if p.strip()]))

    for p in products:
        product_counts[p] = product_counts.get(p,0) + 1

    for p1,p2 in itertools.combinations(sorted(products), 2):
        all_pairs.append({'Product_1': p1, 'Product_2' : p2})


In [25]:
df = pd.DataFrame(all_pairs)
rules = df.groupby(['Product_1', 'Product_2']).size().reset_index(name='Support')
min_support= 15
rules = rules[rules['Support'] >= min_support]
rules.head()

,Product_1,Product_2,Support
1460,0% Fat Free Organic Milk,Bag of Organic Bananas,16
1469,0% Fat Free Organic Milk,Banana,51
2099,0% Fat Free Organic Milk,Large Lemon,20
2358,0% Fat Free Organic Milk,Organic Blueberries,16
2512,0% Fat Free Organic Milk,Organic Strawberries,18


In [28]:
min_confidence = 45.0
rules['Count_1'] = rules['Product_1'].map(product_counts)
rules['Count_2'] = rules['Product_2'].map(product_counts)

rules['Conf_1_to_2'] = (rules['Support'] / rules['Count_1']) * 100
rules['Conf_2_to_1'] = (rules['Support'] / rules['Count_2']) * 100

print('Final results: ')

for index, row in rules.iterrows():
    if row['Conf_1_to_2'] >= min_confidence:
        print(f"Product_1 {row['Product_1']} => {row['Product_2']} ({row['Conf_1_to_2']:.2f}% confidence), {row['Support']} support")
    if row['Conf_2_to_1'] >= min_confidence:
        print(f"Product_2 {row['Product_2']} => {row['Product_1']} ({row['Conf_2_to_1']:.2f}% confidence),{row['Support']} support")

Final results: 
Product_1 0% Greek, Blueberry on the Bottom Yogurt => Nonfat Strawberry With Fruit On The Bottom Greek Yogurt (56.41% confidence), 22 support
Product_1 100% Grass-Fed No-Grain Strawberry Yogurt => Organic Grassmilk Yogurt Wild Blueberry (63.64% confidence), 21 support
Product_2 Dark Chocolate Chia Bar => Acai Berry Chia Bar (53.33% confidence),16 support
Product_1 All Natural Whole Strawberries => Banana (48.89% confidence), 22 support
Product_2 Almond Milk Peach Yogurt => Almond Milk Blueberry Yogurt (48.75% confidence),78 support
Product_1 Amber Ale => India Pale Ale (60.00% confidence), 15 support
Product_2 Antioxidant Infusions Ipanema Pomegranate Beverage => Antioxidant Infusions Brasilia Blueberry (45.45% confidence),15 support
Product_1 Apple Blueberry Fruit Yogurt Smoothie => Organic Fruit Yogurt Smoothie Mixed Berry (48.94% confidence), 23 support
Product_2 Organic Fruit Yogurt Smoothie Peach Banana => Apple Blueberry Fruit Yogurt Smoothie (45.71% confidence),1

In [29]:
print(f'Довжина списку - {len(lines)}')
print(f'Кількість унікальних товарів - {len(product_counts)}')
unique_pairs = set((p['Product_1'], p['Product_2']) for p in all_pairs)
print(f'Знайдено {len(unique_pairs)} пар товарів із {len(lines)} замовлень')

Довжина списку - 131209
Кількість унікальних товарів - 39123
Знайдено 5711806 пар товарів із 131209 замовлень
